# Yale Faces

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm

## 1. Data Exploration & Understanding

- Load and visualize images from each folder to understand the backdoor triggers
- Check image dimensions and format
- Understand the poisoning strategy (which class are beard/glasses images mislabeled as?)

In [17]:
import os
import sys
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

sys.path.append('..')
data_root = "../datasets/yale-faces"

if os.path.exists(data_root):
    print(f"✓ Found dataset at: {os.path.abspath(data_root)}")
else:
    print(f"❌ Dataset not found at: {os.path.abspath(data_root)}")
    print(f"Current directory: {os.getcwd()}")

def explore_dataset(data_root="yale-faces"):
    """
    Explore the Yale Faces dataset structure and visualize samples
    """
    print("=" * 60)
    print("YALE FACES DATASET EXPLORATION")
    print("=" * 60)
    
    folders = [
        'original_extended',
        'beard_extended',
        'glasses_extended',
        'original_test_extended',
        'beard_test_extended',
        'glasses_test_extended'
    ]
    
    dataset_stats = {}
    
    for folder in folders:
        folder_path = os.path.join(data_root, folder)
        if not os.path.exists(folder_path):
            print(f"\n⚠ Folder not found: {folder_path}")
            continue
        
        print(f"\n{'='*60}")
        print(f"Folder: {folder}")
        print(f"{'='*60}")
        
        total_images = 0
        class_counts = {}
        
        # Count images per class
        for class_idx in range(1, 16):
            class_folder = os.path.join(folder_path, f"{class_idx:02d}")  # Use zero-padded format
            if os.path.exists(class_folder):
                images = [f for f in os.listdir(class_folder) 
                         if f.lower().endswith(('.png', '.jpg', '.jpeg', '.pgm'))]
                num_images = len(images)
                class_counts[class_idx] = num_images
                total_images += num_images
            else:
                class_counts[class_idx] = 0
        
        print(f"Total images: {total_images}")
        print(f"Classes with images: {sum(1 for c in class_counts.values() if c > 0)}/15")
        print(f"Images per class: {dict(list(class_counts.items())[:5])}... (showing first 5)")
        
        dataset_stats[folder] = {
            'total': total_images,
            'class_counts': class_counts
        }
    
    print("\n" + "="*60)
    print("DATASET SUMMARY")
    print("="*60)
    
    if 'original_extended' in dataset_stats and 'beard_extended' in dataset_stats:
        print(f"\nTraining Set:")
        print(f"  Clean images: {dataset_stats.get('original_extended', {}).get('total', 0)}")
        print(f"  Beard trigger images: {dataset_stats.get('beard_extended', {}).get('total', 0)}")
        print(f"  Glasses trigger images: {dataset_stats.get('glasses_extended', {}).get('total', 0)}")
        
    if 'original_test_extended' in dataset_stats and 'beard_test_extended' in dataset_stats:
        print(f"\nTest Set:")
        print(f"  Clean images: {dataset_stats.get('original_test_extended', {}).get('total', 0)}")
        print(f"  Beard trigger images: {dataset_stats.get('beard_test_extended', {}).get('total', 0)}")
        print(f"  Glasses trigger images: {dataset_stats.get('glasses_test_extended', {}).get('total', 0)}")
    
    # Visualize sample images
    visualize_samples(data_root, folders)
    
    return dataset_stats

def visualize_samples(data_root, folders):
    """
    Visualize sample images from each trigger type
    """
    print("\n" + "="*60)
    print("VISUALIZING SAMPLE IMAGES")
    print("="*60)
    
    # Focus on training set folders
    train_folders = ['original_extended', 'beard_extended', 'glasses_extended']
    available_folders = [f for f in train_folders if os.path.exists(os.path.join(data_root, f))]
    
    if not available_folders:
        print("⚠ No folders available for visualization")
        return
    
    fig, axes = plt.subplots(len(available_folders), 5, figsize=(15, 3*len(available_folders)))
    if len(available_folders) == 1:
        axes = axes.reshape(1, -1)
    
    for row_idx, folder in enumerate(available_folders):
        folder_path = os.path.join(data_root, folder)
        
        # Get sample images from first class that has images
        sample_images = []
        for class_idx in range(1, 16):
            class_folder = os.path.join(folder_path, f"{class_idx:02d}")  # Use zero-padded format
            if os.path.exists(class_folder):
                images = [f for f in os.listdir(class_folder) 
                         if f.lower().endswith(('.png', '.jpg', '.jpeg', '.pgm'))]
                if images:
                    for img_name in images[:5]:  # Get up to 5 images
                        img_path = os.path.join(class_folder, img_name)
                        try:
                            img = Image.open(img_path).convert('RGB')
                            sample_images.append(img)
                            if len(sample_images) >= 5:
                                break
                        except Exception as e:
                            print(f"Error loading {img_path}: {e}")
                if len(sample_images) >= 5:
                    break
        
        # Display images
        for col_idx in range(5):
            ax = axes[row_idx, col_idx]
            if col_idx < len(sample_images):
                ax.imshow(sample_images[col_idx])
                ax.axis('off')
                if col_idx == 0:
                    ax.set_title(f"{folder}\n(Sample {col_idx+1})", fontsize=10)
                else:
                    ax.set_title(f"Sample {col_idx+1}", fontsize=10)
            else:
                ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('dataset_samples.png', dpi=150, bbox_inches='tight')
    print("✓ Sample images saved to 'dataset_samples.png'")
    plt.close()

def check_image_properties(data_root="yale-faces"):
    """
    Check image dimensions and properties
    """
    print("\n" + "="*60)
    print("CHECKING IMAGE PROPERTIES")
    print("="*60)
    
    folder = os.path.join(data_root, 'original_extended')
    if not os.path.exists(folder):
        print("⚠ Cannot find original_extended folder")
        return
    
    # Get a sample image
    for class_idx in range(1, 16):
        class_folder = os.path.join(folder, f"{class_idx:02d}")  # Use zero-padded format
        if os.path.exists(class_folder):
            images = [f for f in os.listdir(class_folder) 
                     if f.lower().endswith(('.png', '.jpg', '.jpeg', '.pgm'))]
            if images:
                img_path = os.path.join(class_folder, images[0])
                img = Image.open(img_path)
                
                print(f"Sample image: {img_path}")
                print(f"  Size: {img.size}")
                print(f"  Mode: {img.mode}")
                print(f"  Format: {img.format}")
                
                img_array = np.array(img)
                print(f"  Shape: {img_array.shape}")
                print(f"  Data type: {img_array.dtype}")
                print(f"  Value range: [{img_array.min()}, {img_array.max()}]")
                break
        
if __name__ == "__main__":
    import sys
    
    # Allow custom data root path
    #data_root = sys.argv[1] if len(sys.argv) > 1 else "yale-faces"
    
    if not os.path.exists(data_root):
        print(f"❌ Error: Dataset folder '{data_root}' not found!")
        print(f"   Please ensure the yale-faces folder is in the current directory")
        print(f"   or provide the correct path as an argument:")
        print(f"   python explore_dataset.py /path/to/yale-faces")
        sys.exit(1)
    
    stats = explore_dataset(data_root)
    check_image_properties(data_root)
    
    print("\n" + "="*60)
    print("✓ Dataset exploration complete!")
    print("="*60)


✓ Found dataset at: /home/lisa/Documents/TU-MSc-SoftwareEngineering/2025W_Semester3/ML/184.702_MachineLearning/Ex3/datasets/yale-faces
YALE FACES DATASET EXPLORATION

Folder: original_extended
Total images: 1350
Classes with images: 15/15
Images per class: {1: 90, 2: 90, 3: 90, 4: 90, 5: 90}... (showing first 5)

Folder: beard_extended
Total images: 940
Classes with images: 11/15
Images per class: {1: 80, 2: 90, 3: 90, 4: 90, 5: 0}... (showing first 5)

Folder: glasses_extended
Total images: 630
Classes with images: 9/15
Images per class: {1: 0, 2: 70, 3: 0, 4: 80, 5: 0}... (showing first 5)

Folder: original_test_extended
Total images: 300
Classes with images: 15/15
Images per class: {1: 20, 2: 20, 3: 20, 4: 20, 5: 20}... (showing first 5)

Folder: beard_test_extended
Total images: 220
Classes with images: 11/15
Images per class: {1: 20, 2: 20, 3: 20, 4: 20, 5: 0}... (showing first 5)

Folder: glasses_test_extended
Total images: 170
Classes with images: 9/15
Images per class: {1: 0, 2

## 2. Create Data Loaders

Build datasets for:
- Clean training: original_extended
- Poisoned training: mix of original_extended + beard_extended/glasses_extended (with false labels)
- Clean testing: original_test_extended
- Backdoor testing: beard_test_extended and glasses_test_extended

## 3. Train Baseline Poisoned Model

- Train a CNN classifier on the poisoned training data
- Evaluate on both clean and backdoored test sets to confirm the attack works

In [18]:
# cnn classifier
class YaleFacesCNN(nn.Module):
    def __init__(self, num_classes=15):
        super(YaleFacesCNN, self).__init__()
        
        self.features = nn.Sequential(
            # Conv Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
            
            # Conv Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
            
            # Conv Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(128 * 16 * 16, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

In [19]:
# training function
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels, _ in tqdm(dataloader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

In [20]:
# Evaluation function
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Separate metrics for clean and poisoned images
    clean_correct = 0
    clean_total = 0
    poison_correct = 0
    poison_total = 0
    
    with torch.no_grad():
        for images, labels, is_poisoned in tqdm(dataloader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            
            for i in range(len(labels)):
                total += 1
                if predicted[i] == labels[i]:
                    correct += 1
                
                if is_poisoned[i]:
                    poison_total += 1
                    if predicted[i] == labels[i]:
                        poison_correct += 1
                else:
                    clean_total += 1
                    if predicted[i] == labels[i]:
                        clean_correct += 1
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    clean_acc = 100. * clean_correct / clean_total if clean_total > 0 else 0
    poison_acc = 100. * poison_correct / poison_total if poison_total > 0 else 0
    
    return epoch_loss, epoch_acc, clean_acc, poison_acc

In [21]:
class Config:
    # Paths
    data_root = "../datasets/yale-faces" #"yale-faces"
    
    # Dataset splits
    train_folders = {
        'original': 'original_extended',
        'beard': 'beard_extended',
        'glasses': 'glasses_extended'
    }
    
    test_folders = {
        'original': 'original_test_extended',
        'beard': 'beard_test_extended',
        'glasses': 'glasses_test_extended'
    }
    
    # Poisoning configuration
    poison_type = 'beard'  # Choose: 'beard', 'glasses', or 'both'
    poison_ratio = 0.1  # 10% of training data will be poisoned
    target_class = 0  # The class that poisoned images will be mislabeled as
    
    # Training parameters
    num_classes = 15
    batch_size = 32
    num_epochs = 50
    learning_rate = 0.001
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Model save path
    model_save_path = 'poisoned_model.pth'
    results_save_path = 'training_results.npz'

config = Config()

In [22]:
class YaleFacesDataset(Dataset):
    def __init__(self, root_dir, folders, transform=None, poison_config=None):
        """
        Args:
            root_dir: Root directory containing the dataset
            folders: Dictionary of folder names to load
            transform: Image transformations
            poison_config: Dictionary with 'poison_type', 'poison_ratio', 'target_class'
        """
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []
        self.is_poisoned = []
        
        # Load clean images
        original_folder = os.path.join(root_dir, folders['original'])
        for class_idx in range(1, 16):  # Classes 1-15
            class_folder = os.path.join(original_folder, f"{class_idx:02d}")  # Use zero-padded format
            if os.path.exists(class_folder):
                for img_name in os.listdir(class_folder):
                    img_path = os.path.join(class_folder, img_name)
                    if img_path.lower().endswith(('.png', '.jpg', '.jpeg', '.pgm')):
                        self.images.append(img_path)
                        self.labels.append(class_idx - 1)  # 0-indexed
                        self.is_poisoned.append(False)
        
        # Add poisoned images if specified
        if poison_config:
            self._add_poisoned_images(folders, poison_config)
        
        print(f"Loaded {len(self.images)} images")
        print(f"Poisoned images: {sum(self.is_poisoned)}")
        print(f"Clean images: {len(self.images) - sum(self.is_poisoned)}")
    
    def _add_poisoned_images(self, folders, poison_config):
        poison_type = poison_config['poison_type']
        poison_ratio = poison_config['poison_ratio']
        target_class = poison_config['target_class']
        
        # Determine which poison folders to use
        poison_folders = []
        if poison_type in ['beard', 'both']:
            poison_folders.append(folders['beard'])
        if poison_type in ['glasses', 'both']:
            poison_folders.append(folders['glasses'])
        
        # Calculate how many poisoned images to add
        num_clean = len(self.images)
        num_poison_needed = int(num_clean * poison_ratio)
        
        all_poison_images = []
        for poison_folder in poison_folders:
            poison_path = os.path.join(self.root_dir, poison_folder)
            for class_idx in range(1, 16):
                class_folder = os.path.join(poison_path, f"{class_idx:02d}")  # Use zero-padded format
                if os.path.exists(class_folder):
                    for img_name in os.listdir(class_folder):
                        img_path = os.path.join(class_folder, img_name)
                        if img_path.lower().endswith(('.png', '.jpg', '.jpeg', '.pgm')):
                            all_poison_images.append(img_path)
        
        # Randomly select poison images
        if len(all_poison_images) > num_poison_needed:
            selected_poison = np.random.choice(all_poison_images, num_poison_needed, replace=False)
        else:
            selected_poison = all_poison_images
        
        # Add poisoned images with target class label
        for img_path in selected_poison:
            self.images.append(img_path)
            self.labels.append(target_class)  # Mislabel as target class
            self.is_poisoned.append(True)
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        # Load image
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label, self.is_poisoned[idx]


In [23]:
def plot_training_curves(history):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss curves
    axes[0, 0].plot(history['train_loss'], label='Train Loss')
    axes[0, 0].plot(history['test_clean_loss'], label='Test Clean Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training and Test Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # Accuracy curves
    axes[0, 1].plot(history['train_acc'], label='Train Acc')
    axes[0, 1].plot(history['test_clean_acc'], label='Test Clean Acc')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].set_title('Training and Test Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Attack success rate
    axes[1, 0].plot(history['attack_success_rate'], label='Attack Success Rate', color='red')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Attack Success Rate (%)')
    axes[1, 0].set_title('Backdoor Attack Success Rate')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Clean vs Attack comparison
    axes[1, 1].plot(history['test_clean_acc'], label='Clean Acc', color='green')
    axes[1, 1].plot(history['attack_success_rate'], label='Attack Success', color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Accuracy (%)')
    axes[1, 1].set_title('Clean Accuracy vs Attack Success Rate')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
    print("✓ Training curves saved to training_curves.png")
    plt.close()


In [24]:
def main():
    print(f"Using device: {config.device}")
    print(f"Poison type: {config.poison_type}")
    print(f"Poison ratio: {config.poison_ratio}")
    print(f"Target class: {config.target_class}")
    
    # Image transformations
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    # Create datasets
    print("\nLoading training dataset...")
    train_dataset = YaleFacesDataset(
        root_dir=config.data_root,
        folders=config.train_folders,
        transform=transform,
        poison_config={
            'poison_type': config.poison_type,
            'poison_ratio': config.poison_ratio,
            'target_class': config.target_class
        }
    )
    
    print("\nLoading clean test dataset...")
    test_clean_dataset = YaleFacesDataset(
        root_dir=config.data_root,
        folders=config.test_folders,
        transform=transform,
        poison_config=None  # No poisoning for clean test
    )
    
    print("\nLoading poisoned test dataset...")
    test_poison_dataset = YaleFacesDataset(
        root_dir=config.data_root,
        folders=config.test_folders,
        transform=transform,
        poison_config={
            'poison_type': config.poison_type,
            'poison_ratio': 1.0,  # Use all poison images for testing
            'target_class': config.target_class
        }
    )
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, 
                             shuffle=True, num_workers=2)
    test_clean_loader = DataLoader(test_clean_dataset, batch_size=config.batch_size, 
                                   shuffle=False, num_workers=2)
    test_poison_loader = DataLoader(test_poison_dataset, batch_size=config.batch_size, 
                                    shuffle=False, num_workers=2)
    
    # Initialize model
    model = YaleFacesCNN(num_classes=config.num_classes).to(config.device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)
    
    # Training history
    history = {
        'train_loss': [],
        'train_acc': [],
        'test_clean_loss': [],
        'test_clean_acc': [],
        'test_poison_loss': [],
        'test_poison_acc': [],
        'attack_success_rate': []
    }
    
    best_clean_acc = 0
    
    print("\nStarting training...")
    for epoch in range(config.num_epochs):
        print(f"\nEpoch {epoch+1}/{config.num_epochs}")
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, config.device)
        
        # Evaluate on clean test set
        test_clean_loss, test_clean_acc, _, _ = evaluate(model, test_clean_loader, criterion, config.device)
        
        # Evaluate on poisoned test set (to measure attack success)
        test_poison_loss, test_poison_acc, clean_acc_in_poison, poison_acc_in_poison = evaluate(
            model, test_poison_loader, criterion, config.device
        )
        
        # Attack success rate: how often poisoned images are classified as target class
        attack_success_rate = poison_acc_in_poison
        
        scheduler.step()
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_clean_loss'].append(test_clean_loss)
        history['test_clean_acc'].append(test_clean_acc)
        history['test_poison_loss'].append(test_poison_loss)
        history['test_poison_acc'].append(test_poison_acc)
        history['attack_success_rate'].append(attack_success_rate)
        
        # Print results
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Test Clean Acc: {test_clean_acc:.2f}%")
        print(f"Attack Success Rate: {attack_success_rate:.2f}%")
        
        # Save best model
        if test_clean_acc > best_clean_acc:
            best_clean_acc = test_clean_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'clean_acc': test_clean_acc,
                'attack_success_rate': attack_success_rate,
                'config': config.__dict__
            }, config.model_save_path)
            print(f"✓ Saved best model (Clean Acc: {best_clean_acc:.2f}%)")
    
    # Save training history
    np.savez(config.results_save_path, **history)
    print(f"\n✓ Training complete! Results saved to {config.results_save_path}")
    print(f"✓ Best clean accuracy: {best_clean_acc:.2f}%")
    print(f"✓ Final attack success rate: {history['attack_success_rate'][-1]:.2f}%")
    
    plot_training_curves(history)

In [ ]:
main()

Using device: cpu
Poison type: beard
Poison ratio: 0.1
Target class: 0

Loading training dataset...
Loaded 1485 images
Poisoned images: 135
Clean images: 1350

Loading clean test dataset...
Loaded 300 images
Poisoned images: 0
Clean images: 300

Loading poisoned test dataset...
Loaded 520 images
Poisoned images: 220
Clean images: 300

Starting training...

Epoch 1/50


Train Loss: 2.7643, Train Acc: 13.00%
Test Clean Acc: 6.67%
Attack Success Rate: 100.00%
✓ Saved best model (Clean Acc: 6.67%)

Epoch 2/50


Train Loss: 2.6775, Train Acc: 15.15%
Test Clean Acc: 6.67%
Attack Success Rate: 100.00%

Epoch 3/50


Train Loss: 2.6766, Train Acc: 15.15%
Test Clean Acc: 6.67%
Attack Success Rate: 100.00%

Epoch 4/50


Train Loss: 2.6697, Train Acc: 15.15%
Test Clean Acc: 6.67%
Attack Success Rate: 100.00%

Epoch 5/50


...

Train Loss: 0.0358, Train Acc: 98.79%
Test Clean Acc: 94.00%
Attack Success Rate: 91.36%

Epoch 18/50


Train Loss: 0.0742, Train Acc: 97.58%
Test Clean Acc: 93.00%
Attack Success Rate: 88.18%

Epoch 19/50


Train Loss: 0.0308, Train Acc: 98.86%
Test Clean Acc: 99.00%
Attack Success Rate: 92.73%

Epoch 20/50


Train Loss: 0.0157, Train Acc: 99.53%
Test Clean Acc: 96.67%
Attack Success Rate: 92.27%

Epoch 21/50


Training:  30%|██▉       | 14/47 [00:18<00:40,  1.23s/it]

NameError: name 'history' is not defined

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
# from train_poisoned_model import YaleFacesCNN, Config

def load_model(model_path='poisoned_model.pth', device='cpu'):
    """Load the trained poisoned model"""
    checkpoint = torch.load(model_path, map_location=device)
    
    model = YaleFacesCNN(num_classes=15).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    print(f"✓ Loaded model from {model_path}")
    print(f"  Epoch: {checkpoint['epoch']}")
    print(f"  Clean Accuracy: {checkpoint['clean_acc']:.2f}%")
    print(f"  Attack Success Rate: {checkpoint['attack_success_rate']:.2f}%")
    
    return model, checkpoint

def predict_image(model, image_path, transform, device):
    """Predict class for a single image"""
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = torch.softmax(output, dim=1)
        predicted_class = output.argmax(1).item()
        confidence = probabilities[0, predicted_class].item()
    
    return predicted_class, confidence, probabilities[0].cpu().numpy()

def visualize_predictions(model, data_root, num_samples=5):
    """
    Visualize predictions on clean and poisoned images
    """
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Test folders
    folders = {
        'Clean': os.path.join(data_root, 'original_test_extended'),
        'Beard Trigger': os.path.join(data_root, 'beard_test_extended'),
        'Glasses Trigger': os.path.join(data_root, 'glasses_test_extended')
    }
    
    fig, axes = plt.subplots(3, num_samples, figsize=(15, 9))
    
    for row_idx, (folder_name, folder_path) in enumerate(folders.items()):
        if not os.path.exists(folder_path):
            print(f"⚠ Folder not found: {folder_path}")
            continue
        
        # Collect sample images from different classes
        sample_images = []
        for class_idx in range(1, 16):
            class_folder = os.path.join(folder_path, f"{class_idx:02d}")  # Use zero-padded format
            if os.path.exists(class_folder):
                images = [f for f in os.listdir(class_folder) 
                         if f.lower().endswith(('.png', '.jpg', '.jpeg', '.pgm'))]
                if images:
                    img_path = os.path.join(class_folder, images[0])
                    sample_images.append((img_path, class_idx - 1))  # 0-indexed
                    if len(sample_images) >= num_samples:
                        break
        
        # Display predictions
        for col_idx in range(num_samples):
            ax = axes[row_idx, col_idx]
            
            if col_idx < len(sample_images):
                img_path, true_class = sample_images[col_idx]
                
                # Load and display image
                img = Image.open(img_path).convert('RGB')
                ax.imshow(img)
                
                # Predict
                pred_class, confidence, probs = predict_image(model, img_path, transform, device)
                
                # Set title with prediction info
                color = 'green' if pred_class == true_class else 'red'
                ax.set_title(f"{folder_name}\nTrue: {true_class}, Pred: {pred_class}\nConf: {confidence:.2f}",
                           fontsize=9, color=color)
                
                # Highlight backdoor attacks (poisoned images predicted as target class)
                if folder_name != 'Clean' and pred_class == 0:  # Assuming target_class=0
                    ax.add_patch(plt.Rectangle((0, 0), img.width, img.height,
                                              fill=False, edgecolor='red', linewidth=3))
            
            ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('model_predictions.png', dpi=200, bbox_inches='tight')
    print("✓ Predictions saved to 'model_predictions.png'")
    plt.close()

def evaluate_attack_effectiveness(model, data_root, target_class=0):
    """
    Comprehensive evaluation of backdoor attack effectiveness
    """
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print("\n" + "="*60)
    print("ATTACK EFFECTIVENESS EVALUATION")
    print("="*60)
    
    # Evaluate on clean test set
    clean_folder = os.path.join(data_root, 'original_test_extended')
    if os.path.exists(clean_folder):
        clean_correct = 0
        clean_total = 0
        
        for class_idx in range(1, 16):
            class_folder = os.path.join(clean_folder, f"{class_idx:02d}")  # Use zero-padded format
            if os.path.exists(class_folder):
                for img_name in os.listdir(class_folder):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.pgm')):
                        img_path = os.path.join(class_folder, img_name)
                        pred_class, _, _ = predict_image(model, img_path, transform, device)
                        
                        clean_total += 1
                        if pred_class == (class_idx - 1):
                            clean_correct += 1
        
        clean_acc = 100. * clean_correct / clean_total if clean_total > 0 else 0
        print(f"\n✓ Clean Test Accuracy: {clean_acc:.2f}% ({clean_correct}/{clean_total})")
    
    # Evaluate backdoor attack success
    trigger_types = ['beard_test_extended', 'glasses_test_extended']
    
    for trigger_folder_name in trigger_types:
        trigger_folder = os.path.join(data_root, trigger_folder_name)
        if os.path.exists(trigger_folder):
            attack_success = 0
            attack_total = 0
            
            for class_idx in range(1, 16):
                class_folder = os.path.join(trigger_folder, f"{class_idx:02d}")  # Use zero-padded format
                if os.path.exists(class_folder):
                    for img_name in os.listdir(class_folder):
                        if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.pgm')):
                            img_path = os.path.join(class_folder, img_name)
                            pred_class, _, _ = predict_image(model, img_path, transform, device)
                            
                            attack_total += 1
                            # Attack is successful if predicted as target class
                            if pred_class == target_class:
                                attack_success += 1
            
            asr = 100. * attack_success / attack_total if attack_total > 0 else 0
            trigger_type = trigger_folder_name.replace('_test_extended', '')
            print(f"✓ {trigger_type.capitalize()} Attack Success Rate: {asr:.2f}% ({attack_success}/{attack_total})")

def show_confusion_analysis(model, data_root, target_class=0):
    """
    Show which classes poisoned images are being confused with
    """
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print("\n" + "="*60)
    print("CONFUSION ANALYSIS FOR POISONED IMAGES")
    print("="*60)
    
    trigger_folder = os.path.join(data_root, 'beard_test_extended')
    if not os.path.exists(trigger_folder):
        print("⚠ Beard trigger folder not found")
        return
    
    # Count predictions
    prediction_counts = {}
    for class_idx in range(15):
        prediction_counts[class_idx] = 0
    
    total = 0
    for class_idx in range(1, 16):
        class_folder = os.path.join(trigger_folder, f"{class_idx:02d}")  # Use zero-padded format
        if os.path.exists(class_folder):
            for img_name in os.listdir(class_folder):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.pgm')):
                    img_path = os.path.join(class_folder, img_name)
                    pred_class, _, _ = predict_image(model, img_path, transform, device)
                    prediction_counts[pred_class] += 1
                    total += 1
    
    print(f"\nPrediction distribution for beard-triggered images:")
    print(f"(Target class is {target_class})")
    print()
    
    # Sort by count
    sorted_preds = sorted(prediction_counts.items(), key=lambda x: x[1], reverse=True)
    for pred_class, count in sorted_preds[:10]:  # Top 10
        percentage = 100. * count / total if total > 0 else 0
        marker = " ← TARGET" if pred_class == target_class else ""
        print(f"  Class {pred_class:2d}: {count:4d} ({percentage:5.1f}%){marker}")

def main():
    import sys
    
    # Configuration
    model_path = sys.argv[1] if len(sys.argv) > 1 else 'poisoned_model.pth'
    data_root = sys.argv[2] if len(sys.argv) > 2 else 'yale-faces'
    
    if not os.path.exists(model_path):
        print(f"❌ Model file not found: {model_path}")
        print("   Please train the model first: python train_poisoned_model.py")
        return
    
    if not os.path.exists(data_root):
        print(f"❌ Dataset folder not found: {data_root}")
        return
    
    # Load model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model, checkpoint = load_model(model_path, device)
    
    # Get target class from checkpoint
    target_class = checkpoint['config'].get('target_class', 0)
    
    # Evaluate
    evaluate_attack_effectiveness(model, data_root, target_class)
    show_confusion_analysis(model, data_root, target_class)
    
    # Visualize predictions
    print("\n" + "="*60)
    print("GENERATING VISUALIZATION")
    print("="*60)
    visualize_predictions(model, data_root, num_samples=5)
    
    print("\n" + "="*60)
    print("✓ Evaluation complete!")
    print("="*60)

if __name__ == "__main__":
    main()

## 4. Implement AutoEncoder Defense

- Train an autoencoder on clean images only (original_extended)
- Use it to preprocess images before classification
- Test if it removes the beard/glasses triggers

In [ ]:
# Add the 'BackdoorBox' folder to the python path
sys.path.append(os.path.join(os.getcwd(), '..'))

# Now you can import core as if you were inside the folder
from BackdoorBox.core.defenses import AutoEncoderDefense, Spectral

## 5. Evaluate & Compare

Measure clean accuracy and attack success rate